# 06 Mood Analysis with GoEmotions

Loads GoEmotions-style TSV data, maps numeric labels using `emotions.txt`, visualizes emotion distribution, optionally creates word clouds, and shows sample text per emotion.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
except Exception:
    sns = None

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ai').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'ai' / 'datasets'
PREPROCESSING_DIR = PROJECT_ROOT / 'ai' / 'preprocessing'
MODEL_DIR = PROJECT_ROOT / 'ai' / 'models'

CSV_TSV_EXTENSIONS = {'.csv', '.tsv'}

def discover_data_files(data_dir=DATA_DIR):
    return sorted(p for p in data_dir.rglob('*') if p.suffix.lower() in CSV_TSV_EXTENSIONS)

def read_dataset(path):
    sep = '\t' if path.suffix.lower() == '.tsv' else ','
    return pd.read_csv(path, sep=sep)

def safe_name(path):
    return path.relative_to(DATA_DIR).as_posix().replace('/', '__')

def display_heading(text):
    try:
        from IPython.display import display, Markdown
        display(Markdown(f'### {text}'))
    except Exception:
        print(f'\n### {text}')

from collections import Counter
try:
    from wordcloud import WordCloud
except Exception:
    WordCloud = None

def find_goemotions_files():
    train = next(DATA_DIR.rglob('train.tsv'), None)
    emotions = next(DATA_DIR.rglob('emotions.txt'), None)
    return train, emotions

def load_emotion_names(path):
    return [line.strip() for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def parse_labels(value, names):
    labels = []
    for part in str(value).replace(',', ' ').split():
        if part.isdigit():
            idx = int(part)
            labels.append(names[idx] if idx < len(names) else f'label_{idx}')
    return labels or ['unknown']


In [ ]:
train_path, emotions_path = find_goemotions_files()
if not train_path or not emotions_path:
    print('GoEmotions train.tsv or emotions.txt was not found under ai/datasets.')
else:
    try:
        names = load_emotion_names(emotions_path)
        raw = pd.read_csv(train_path, sep='\t', header=None)
        if raw.shape[1] >= 2:
            raw = raw.rename(columns={0: 'text', 1: 'labels'})
        else:
            raise ValueError('train.tsv must contain at least text and label columns')
        raw['emotion_list'] = raw['labels'].apply(lambda x: parse_labels(x, names))
        exploded = raw.explode('emotion_list').rename(columns={'emotion_list': 'emotion'})
        display(raw.head())
        distribution = exploded['emotion'].value_counts()
        display(distribution.to_frame('count'))
        distribution.head(20).sort_values().plot(kind='barh', figsize=(9, 6), title='Top emotions')
        plt.tight_layout(); plt.show()
        if WordCloud:
            text = ' '.join(raw['text'].dropna().astype(str).tolist())
            if text.strip():
                wc = WordCloud(width=1000, height=500, background_color='white').generate(text)
                plt.figure(figsize=(12, 6)); plt.imshow(wc); plt.axis('off'); plt.title('GoEmotions word cloud'); plt.show()
        else:
            print('wordcloud is not installed; skipping word cloud generation.')
        display_heading('Sample text per emotion')
        samples = exploded.groupby('emotion')['text'].apply(lambda s: s.dropna().astype(str).head(3).tolist()).reset_index()
        display(samples)
    except Exception as exc:
        print(f'Mood analysis failed: {exc}')
